## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | Inspect Google Drive Model Files |
| Model / workflow | SE-ResNeXt-50 32x4d |
| Input | not reported |
| Loss | not reported |
| Training / pipeline | YOLO detection/ROI workflow |
| Result | No executed result output was recorded. |


# Inspect Google Drive Model Files

This notebook reads an old Google Drive model folder and reports its structure and checkpoint metadata. It is read-only: it never creates, moves, renames, deletes, or overwrites files.

Use the report to decide how to organize the folder manually later.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
from collections import defaultdict

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Change only this name when your models are stored in a different MyDrive folder.
DRIVE_ROOT = Path('/content/drive/MyDrive')
MODEL_FOLDER_NAME = 'Models'
SOURCE_ROOT = DRIVE_ROOT / MODEL_FOLDER_NAME

if not SOURCE_ROOT.is_dir():
    available = sorted(path.name for path in DRIVE_ROOT.iterdir() if path.is_dir()) if DRIVE_ROOT.is_dir() else []
    raise FileNotFoundError(
        f'Model folder not found: {SOURCE_ROOT}. Change MODEL_FOLDER_NAME to the correct folder name. '
        f'Folders directly in MyDrive: {available}'
    )

print('Source:', SOURCE_ROOT)
print('Read-only inventory mode')

Mounted at /content/drive


In [ ]:
def sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def checkpoint_metadata(path: Path) -> dict:
    metadata = {}
    if path.suffix.lower() not in {'.pth', '.pt', '.ckpt'}:
        return metadata
    try:
        import torch
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        if isinstance(checkpoint, dict):
            for key in ('architecture', 'model_name', 'loss_type', 'epoch'):
                value = checkpoint.get(key)
                if value is not None and isinstance(value, (str, int, float, bool)):
                    metadata[key] = value
    except Exception as error:
        metadata['read_error'] = f'{type(error).__name__}: {error}'
    return metadata

def infer_model(path: Path, metadata: dict) -> str:
    text = ' '.join([str(path).lower(), str(metadata.get('model_name', '')).lower(), str(metadata.get('architecture', '')).lower()])
    if 'yolo' in text or path.suffix.lower() == '.pt':
        return 'yolov8'
    if 'resnext' in text or 'seresnext' in text:
        return 'se_resnext50_32x4d'
    if 'densenet' in text:
        return 'densenet121'
    if 'efficientnet' in text:
        return 'efficientnet'
    return 'unknown'

def infer_run_id(path: Path) -> str:
    match = re.search(r'(20\d{2}-\d{2}-\d{2}[^/]*)', str(path))
    return match.group(1).replace(' ', '_') if match else path.parent.name

files = sorted(path for path in SOURCE_ROOT.rglob('*') if path.is_file())
records = []
for source in files:
    metadata = checkpoint_metadata(source)
    records.append({
        'source_relative': str(source.relative_to(SOURCE_ROOT)),
        'source_path': str(source),
        'size_bytes': source.stat().st_size,
        'sha256': sha256(source),
        'suffix': source.suffix.lower(),
        'model': infer_model(source, metadata),
        'run_id': infer_run_id(source),
        'architecture': metadata.get('architecture', ''),
        'model_name': metadata.get('model_name', ''),
        'loss_type': metadata.get('loss_type', ''),
        'epoch': metadata.get('epoch', ''),
        'metadata_error': metadata.get('read_error', ''),
    })

print(f'Files found: {len(records)}')
for model, count in sorted(__import__('collections').Counter(r['model'] for r in records).items()):
    print(f'{model}: {count}')

In [ ]:
# Review duplicates. Same SHA-256 means the file content is identical.
by_hash = defaultdict(list)
for record in records:
    by_hash[record['sha256']].append(record)
duplicates = {digest: items for digest, items in by_hash.items() if len(items) > 1}
print(f'Duplicate content groups: {len(duplicates)}')
for digest, items in list(duplicates.items())[:20]:
    print(digest[:12], [item['source_relative'] for item in items])

checkpoint_suffixes = {'.pth', '.pt', '.ckpt'}
for record in records:
    relative = Path(record['source_relative'])
    is_checkpoint = record['suffix'] in checkpoint_suffixes
    record['is_checkpoint'] = is_checkpoint

checkpoint_records = [r for r in records if r['is_checkpoint']]
print(f'Checkpoint files: {len(checkpoint_records)}')
for record in checkpoint_records:
    print(record['source_relative'], '|', record['model'], '|', record['architecture'], '|', record['loss_type'], '| epoch', record['epoch'])

In [ ]:
# No files are written by this notebook. Use the printed inventory and duplicate list to plan a separate manual migration.
print('Read-only scan complete; no files were created or changed.')

In [ ]:
# Display a compact architecture summary. No manifest is written to Drive.
for record in records:
    record['duplicate_group'] = record['sha256'] if record['sha256'] in duplicates else ''
from collections import Counter
print('Architecture summary:')
for key, count in Counter((r['model'], r['architecture'] or '(unknown)') for r in checkpoint_records).items():
    print(count, key)